#### 7.1. Verifying  libraries

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    try:
        v = version(lib)
        print(f'{lib}: {v}')
    except Exception as e:
        print(f'{lib}: Not found')

reasoning_from_scratch: Not found
torch: 2.4.1+cu124
tokenizers: 0.23.3


#### 7.2. Loading pretrained model

In [11]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using NVIDIA CUDA GPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date


d:\Stuffs\FunProjects\DumLRM\src\dumlrm\chapters\reasoning_from_scratch\ch03.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path

In [12]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 To solve the equation "Half the value of \(3x - 9\) is \(x + 37\)," follow these steps:

1. **Set up the equation based on the given information:**
   \[
   \frac{1}{2} \times (3x - 9) = x + 37
   \]

2. **Eliminate the fraction by multiplying both sides by 2:**
   \[
   2 \times \frac{1}{2} \times (3x - 9) = 2 \times (x + 37)
   \]
   \[
   3x - 9 = 2x + 74
   \]

3. **Isolate the variable \(x\) by subtracting \(2x\) from both sides:**
   \[
   3x - 2x - 9 = 2x - 2x + 74
   \]
   \[
   x - 9 = 74
   \]

4. **Solve for \(x\) by adding 9 to both sides:**
   \[
   x - 9 + 9 = 74 + 9
   \]
   \[
   x = 83
   \]

**Final Answer:**
\[
\boxed{83}
\]

#### 7.3. Tracking GRPO performance metrics

##### 7.3.1 Advantages tracking

In [1]:
import torch

def compute_advantage_stats(rewards_list):
    rewards = torch.tensor(rewards_list)

    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()

    return advantages, adv_avg, adv_std

##### 7.3.2 Entropy tracking

In [4]:
# entropy = -1 * prob * log_prob

# Toy logits
logits = torch.tensor([
    0.6667, -2.0000,  1.3333, -0.0000, -0.6667,  2.0000, -1.3333
])

logprobs = torch.log_softmax(logits, dim=-1)
print("all token logprobs       : ", logprobs)

selected_token = torch.argmax(logprobs)
selected = logprobs[selected_token]
print("selected token           : ", selected_token)
print("selected token logprob   : ", selected)

all token logprobs       :  tensor([-2.0442, -4.7109, -1.3776, -2.7109, -3.3776, -0.7109, -4.0442])
selected token           :  tensor(5)
selected token logprob   :  tensor(-0.7109)


In [7]:
logprobs = torch.log_softmax(logits, dim=-1)
# optimization skipping probs
probs = torch.exp(logprobs)

# entropy
entropy = torch.sum(-1 * probs * logprobs)

print("probs    :", probs)
print("logprobs :", logprobs)
print("entropy  :", entropy)


probs    : tensor([0.1295, 0.0090, 0.2522, 0.0665, 0.0341, 0.4912, 0.0175])
logprobs : tensor([-2.0442, -4.7109, -1.3776, -2.7109, -3.3776, -0.7109, -4.0442])
entropy  : tensor(1.3700)


In [ ]:
# max_entropy = log(vocab_size)
vocab_size = 151_900
torch.log(torch.tensor(vocab_size))

tensor(11.9310)

In [13]:
def sequence_logprob_and_entropy(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)

    sequence_answer_logprob = selected[prompt_len - 1]

    logp_all_steps = torch.sum(sequence_answer_logprob)

    # calculate entropy
    all_answer_logprobs = logprobs[:-1][prompt_len - 1]
    if all_answer_logprobs.numel() == 0:
        entropy_all_steps = logp_all_steps.new_tensor(0.0)
    else:
        all_answer_probs = torch.exp(all_answer_logprobs)
        plogp = all_answer_probs * all_answer_logprobs
        step_entropy = -1 * torch.sum(plogp, dim=-1)
        entropy_all_steps = torch.mean(step_entropy)

    return logp_all_steps, entropy_all_steps

#### 7.4. Stabalizing sequence-level GRPO using clipped policy ratios

In [14]:
# Compute rollouts
# ...

# Compute rewards
rewards = torch.tensor([1., 1., 0., 0.])

# Compute sequence logprobs
logprobs = torch.tensor([-7.9243, -20.1546, -16.6130, -23.3677])

advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

pg_loss = -(advantages.detach() * logprobs).mean()
print("Policy gradient loss:", pg_loss)

Policy gradient loss: tensor(-2.5764)


In [16]:
# ratio = exp(logp_new - logp_old)
# clipping happens on extreme ratios

new_logps = logprobs

# The values of the new_logps are shown side by side in the comments
old_logps = torch.tensor([
    -10.9243,   # -7.9243
    -20.3546,   # -20.1546
    -14.6130,   # -16.6130
    -23.3677,   # -23.3677
])

log_ratio = new_logps - old_logps
ratio = torch.exp(log_ratio)
clip_eps = 10.0

clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)

print("Ratio         : ", ratio)
print("Clipped ratio : ", clipped_ratio)

Ratio         :  tensor([20.0855,  1.2214,  0.1353,  1.0000])
Clipped ratio :  tensor([11.0000,  1.2214,  0.1353,  1.0000])


In [17]:
# treat advantages as fixed learning signals
adv = advantages.detach()

unclipped = ratio * adv
clipped = clipped_ratio * adv

# PPO-style clipping
obj = torch.minimum(unclipped, clipped)

clipped_pg_loss = -torch.mean(obj)
policy_ratio = torch.mean(ratio)

print("Clipping policy gradient loss: ", clipped_pg_loss)
print("Policy ratio: ", policy_ratio)

Clipping policy gradient loss:  tensor(-2.3998)
Policy ratio:  tensor(5.6106)


#### 7.5. Controlling how much the model changes with a KL term

In [38]:
import copy 
from reasoning_from_scratch.ch06 import sample_response, reward_rlvr, sequence_logprob

# kl_coeff = 0.0 deactivate the KL term
kl_coeff = 0.1

# creating reference model
# make a copy of original model and disable gradients so it isn't updated
if kl_coeff:
    ref_model = copy.deepcopy(model).to(device)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)
else:
    ref_model = None

def compute_grpo_loss_kl(
        model, ref_model, tokenizer, example, device,
        num_rollouts=2, max_new_tokens=256,
        temperature=0.8, top_p=0.9, kl_coeff=0.02
):
    assert num_rollouts >= 2 
    roll_logps, roll_ref_logps, roll_rewards, samples = [], [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # 1. generate rollouts
        token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )

        # 2. compute rewards
        reward = reward_rlvr(answer_text, example["solution"])

        # 3. compute log probs with current model
        logp = sequence_logprob(model, token_ids, prompt_len)

        # 4. compute log probs with reference model
        if kl_coeff:
            with torch.no_grad():
                ref_logps = sequence_logprob(ref_model, token_ids, prompt_len)
        else:
            ref_logps = None

        roll_logps.append(logp)
        roll_rewards.append(reward)
        if kl_coeff:
            roll_ref_logps.append(ref_logps)

        samples.append(
            {
                "text": answer_text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # 2. collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # 3. compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # 4. collect all logprobs
    logps = torch.stack(roll_logps)

    # 5. compute KL divergence
    if kl_coeff:
        ref_logps = torch.stack(roll_ref_logps).detach()

    # compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    
    # compute KL loss term
    if kl_coeff:
        kl_loss = kl_coeff * torch.mean(torch.exp(logps - ref_logps)) * (logps - ref_logps) 
    else:
        kl_loss = torch.tensor(0.0, device=device)

    loss = pg_loss + kl_loss

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

#### 7.6. Adding an explicit format reward

In [ ]:
print(tokenizer.encode("<think>"))

[13708, 766, 29]


In [34]:
tokenizer._tok.add_special_tokens(
    ["<tool_response>", "</tool_response>", "<think>", "</think>"]
)


4

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(
    kind="reasoning", tokenizer_only=True, out_dir="qwen3"
)

tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

✓ qwen3\tokenizer-reasoning.json already up-to-date


In [35]:

print(tokenizer.encode("<think>"))
print(tokenizer.encode("</think>"))

[151667]
[151668]


In [36]:
def reward_format(
        token_ids, prompt_len,
        start_think_id = 151667,
        end_think_id = 151668,
):
    try:
        gen = token_ids[prompt_len:].tolist()
        return float(
            gen.index(start_think_id) < gen.index(end_think_id)
        )
    except ValueError:
        return 0.0


In [37]:
prompt = "Calculate ..."
rollout = "Let's ... <think> ... </think> ..."
token_ids = tokenizer.encode(prompt + rollout)

reward_format(
    token_ids=torch.tensor(token_ids),
    prompt_len=len(tokenizer.encode(prompt))
)

1.0

In [41]:
def compute_grpo_loss_kl_format(
        model, ref_model, tokenizer, example, device,
        num_rollouts=2, max_new_tokens=256,
        temperature=0.8, top_p=0.9, kl_coeff=0.02,
        format_reward_weight = 1.0
):
    assert num_rollouts >= 2 
    roll_logps, roll_ref_logps, roll_rewards, samples = [], [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )

        # NEW: COMPUTE REWARD FROM RLVR AND FORMAT
        # compute format_reward if rlvr_reward is correct
        rlvr_reward = reward_rlvr(answer_text, example["solution"])
        format_reward = rlvr_reward * reward_format(token_ids, prompt_len)
        reward = rlvr_reward + format_reward_weight * format_reward 
        
        logp = sequence_logprob(model, token_ids, prompt_len)

        if kl_coeff:
            with torch.no_grad():
                ref_logps = sequence_logprob(ref_model, token_ids, prompt_len)
        else:
            ref_logps = None

        roll_logps.append(logp)
        roll_rewards.append(reward)
        if kl_coeff:
            roll_ref_logps.append(ref_logps)

        samples.append(
            {
                "text": answer_text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    rewards = torch.tensor(roll_rewards, device=device)

    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    logps = torch.stack(roll_logps)

    if kl_coeff:
        ref_logps = torch.stack(roll_ref_logps).detach()

    pg_loss = -(advantages.detach() * logps).mean()
    
    if kl_coeff:
        kl_loss = kl_coeff * torch.mean(torch.exp(logps - ref_logps)) * (logps - ref_logps) 
    else:
        kl_loss = torch.tensor(0.0, device=device)

    loss = pg_loss + kl_loss

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [40]:
def render_prompt_with_think_tokens(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "When solving the problem, first write your reasoning inside <think> and </think> tags.\n"
        "Then write the final result on a new line in the exact format:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template